### IMPORTS

In [1]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
scaler = StandardScaler()

### DATASETS

In [2]:
auto_mpg = pd.read_csv("auto_mpg_cleaned.csv")
auto_mpg = auto_mpg.dropna()
house_price = pd.read_csv('housing.csv')
house_price = house_price.dropna()
house_price = pd.get_dummies(house_price, dtype = int, drop_first = True)
insurance = pd.read_csv('insurance_cat2num.csv')

### CLASS SETUP

In [3]:
#Two Hidden Layer Neural Network
        
'''======================================================================================'''

class OneHiddenLayerNNLeakyRelu(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(OneHiddenLayerNNLeakyRelu, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # Input to hidden layer
        self.leaky_relu = nn.LeakyReLU()              # Activation function
        self.fc2 = nn.Linear(hidden_size, output_size) # Hidden to output layer
        self.batch_size = 32
        self.patience = 25
        self.min_delta = 1e-4
        
    def forward(self, x):
        out = self.fc1(x)   # Linear transformation
        out = self.leaky_relu(out) # Non-linearity
        out = self.fc2(out)  # Linear transformation to output
        return out
    def trainLeakyRelu(self, X_train, y_train, max_epochs=200, learning_rate=0.01):
        self.train()  # Set the model to training mode
        criterion = nn.MSELoss()  # Mean Squared Error Loss
        optimizer = torch.optim.SGD(self.parameters(), lr=learning_rate)  # Stochastic Gradient Descent

        batch_size = self.batch_size

        dataset = TensorDataset(X_train, y_train)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        best_loss = float('inf')
        patience_counter = 0

        for epoch in range(max_epochs):
            epoch_loss = 0.0
            
            for batch_X, batch_y in dataloader:
            
                outputs = self(batch_X)  # Forward pass
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                epoch_loss+=loss.item()
            avg_loss = epoch_loss / len(dataloader)
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}/{max_epochs} | Avg Loss: {avg_loss:.4f}")

            if best_loss - avg_loss > self.min_delta:
                best_loss = avg_loss
                patience_counter = 0
            else:
                patience_counter += 1
            if patience_counter >= self.patience:
                print(f"\nEarly stopping triggered at Epoch {epoch+1}!")
                print(f"Loss hasn't improved by more than {self.min_delta} for {self.patience} consecutive epochs.")
                break
            
        print("Training Complete!")
    def testLeakyRelu(self, X_test, y_test):
        self.eval()  # Set the model to evaluation mode
        loss_fn = nn.MSELoss()
        with torch.no_grad():  # No need to compute gradients during testing
            outputs = self(X_test)  # Forward pass
            test_loss = loss_fn(outputs, y_test)
        print(f"Test Loss: {test_loss.item():.4f}")
        self.train()   
        return(outputs, test_loss.item())

In [4]:
# Two Hidden Layer Neural Network, Leaky ReLU Activation, Early Stopping
        
'''======================================================================================'''

class TwoHiddenLayerNNLeakyRelu(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size):
        super(TwoHiddenLayerNNLeakyRelu, self).__init__()
        
        # Layers
        self.fc1 = nn.Linear(input_size, hidden_size1)   # Input → Hidden 1
        self.fc2 = nn.Linear(hidden_size1, hidden_size2) # Hidden 1 → Hidden 2
        self.fc3 = nn.Linear(hidden_size2, output_size)  # Hidden 2 → Output
        
        # Activation
        self.leaky_relu = nn.LeakyReLU()
        
        # Training params
        self.batch_size = 32
        self.patience = 25
        self.min_delta = 1e-4
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.leaky_relu(out)
        
        out = self.fc2(out)
        out = self.leaky_relu(out)
        
        out = self.fc3(out)
        return out

    def trainLeakyRelu(self, X_train, y_train, max_epochs=200, learning_rate=0.01):
        self.train()
        criterion = nn.MSELoss()
        optimizer = torch.optim.SGD(self.parameters(), lr=learning_rate)

        dataset = TensorDataset(X_train, y_train)
        dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        best_loss = float('inf')
        patience_counter = 0

        for epoch in range(max_epochs):
            epoch_loss = 0.0
            
            for batch_X, batch_y in dataloader:
                outputs = self(batch_X)
                loss = criterion(outputs, batch_y)

                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

                epoch_loss += loss.item()

            avg_loss = epoch_loss / len(dataloader)

            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}/{max_epochs} | Avg Loss: {avg_loss:.4f}")

            if best_loss - avg_loss > self.min_delta:
                best_loss = avg_loss
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= self.patience:
                print(f"\nEarly stopping triggered at Epoch {epoch+1}!")
                print(f"Loss hasn't improved by more than {self.min_delta} for {self.patience} consecutive epochs.")
                break
            
        print("Training Complete!")

    def testLeakyRelu(self, X_test, y_test):
        self.eval()
        loss_fn = nn.MSELoss()

        with torch.no_grad():
            outputs = self(X_test)
            test_loss = loss_fn(outputs, y_test)

        print(f"Test Loss: {test_loss.item():.4f}")
        self.train()
        return (outputs, test_loss.item())

### QOF VARIABLES

In [5]:
def get_qof(y_actual, y_pred, k, mod=None):
    """
    Calculates and compiles a comprehensive set of Quality of Fit (QoF) metrics 
    for a regression model.
    
    This function attempts to extract pre-calculated metrics from a fitted model 
    object (e.g., a Statsmodels result instance) if provided. If the object is not 
    provided or lacks specific metrics, it computes them manually using NumPy.

    Args:
        y_actual (array-like): The actual/true target values.
        y_pred (array-like): The predicted values generated by the model.
        k (int): The number of predictor variables (features) used in the model.
        mod (object, optional): A fitted model object (e.g., from Statsmodels) 
                                that may contain pre-calculated metrics. Defaults to None.

    Returns:
        list: A list containing 15 QoF metrics in the following specific order:
              [R^2, Adj R^2, SST, SSE, SDE, MSE, RMSE, MAE, sMAPE, 
               Number of Observations, DF Model, DF Residuals, F-statistic, AIC, BIC]
    """
    # ==========================================
    # --- Basic Parameters ---
    # ==========================================
    # Get the number of observations (m) in the original dataset
    m = len(y_actual)  

    # ==========================================
    # --- Sum of Squares ---
    # ==========================================
    # Calculate SSE (Sum of Squared Errors)
    sse = getattr(mod, "ssr", np.sum((y_actual - y_pred)**2))

    # Calculate SST (Total Sum of Squares)
    sst = getattr(mod, "centered_tss", np.sum((y_actual - np.mean(y_actual))**2))

    # ==========================================
    # --- R-Squared Metrics ---
    # ==========================================
    # Compute R-squared (Coefficient of Determination)
    r_squared = getattr(mod, "rsquared", 1 - (sse / sst))

    # Calculate Adjusted R-squared (penalizes for adding non-useful predictors)
    adj_r_squared = getattr(mod, "rsquared_adj", 1 - (1 - r_squared) * (m - 1) / (m - k))

    # ==========================================
    # --- Error Metrics ---
    # ==========================================
    # Standard Deviation of the Errors (SDE) / Residual Standard Error
    sde = np.sqrt(getattr(mod, "mse_resid", sse / (m - k)))

    # Calculate Mean Squared Error (MSE)
    mse = getattr(mod, "mse_resid", sse / m)

    # Root Mean Squared Error (RMSE)
    rmse = np.sqrt(getattr(mod, "mse_resid", mse))

    # Mean Absolute Error (MAE)
    mae = getattr(mod, "mae", np.mean(np.abs(y_actual - y_pred)))

    # Symmetric Mean Absolute Percentage Error (sMAPE)
    smape = getattr(mod, "smape", np.mean(2 * np.abs(y_actual - y_pred) / (np.abs(y_actual) + np.abs(y_pred))) * 100)

    # ==========================================
    # --- Statistical Tests & Degrees of Freedom ---
    # ==========================================
    # Calculate F-statistic for overall model significance
    f_stat = getattr(mod, "fvalue", ((sst - sse) / k) / (sse / (m - k)))

    # Get the degrees of freedom for the model (typically equivalent to k)
    df_model = getattr(mod, "df_model", k)

    # Get the degrees of freedom for the residuals (typically m - k)
    df_resid = getattr(mod, "df_resid", m - k)

    # ==========================================
    # --- Information Criteria ---
    # ==========================================
    # Calculate Akaike Information Criterion (AIC)
    aic = getattr(mod, "aic", m * np.log(mse) + 2 * k)

    # Calculate Bayesian Information Criterion (BIC)
    bic = getattr(mod, "bic", m * np.log(mse) + k * np.log(m))

    # ==========================================
    # --- Compile Results ---
    # ==========================================
    # Initialize a list to store exactly 15 elements
    qof = list(range(15))

    qof[0]  = r_squared
    qof[1]  = adj_r_squared
    qof[2]  = sst
    qof[3]  = sse
    qof[4]  = sde
    qof[5]  = mse
    qof[6]  = rmse
    qof[7]  = mae
    qof[8]  = smape
    qof[9]  = m
    qof[10] = df_model
    qof[11] = df_resid
    qof[12] = f_stat
    qof[13] = aic
    qof[14] = bic

    return qof

## IN-SAMPLE

### AUTOMPG

### HOUSE PRICES

### INSURANCE

## TRAIN TEST SPLIT

### AUTOMPG

In [6]:
X = auto_mpg.drop('mpg', axis = 1)
y = auto_mpg[['mpg']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32)
model_sigmoid = TwoHiddenLayerNNLeakyRelu(input_size=X.shape[1], hidden_size1=100, hidden_size2=50, output_size=y.shape[1])
model_sigmoid.trainLeakyRelu(X_train_tensor, y_train_tensor)
(OOS_predictions, OOS_loss) = model_sigmoid.testLeakyRelu(X_test_tensor, y_test_tensor)

y_test_numpy = y_test.to_numpy()
preds_OOS_numpy = OOS_predictions.cpu().numpy()
k = X_train_scaled.shape[1]
qof_OOS = get_qof(y_test_numpy, preds_OOS_numpy, k)
print(qof_OOS)

Epoch 10/200 | Avg Loss: 12.1991
Epoch 20/200 | Avg Loss: 7.0625
Epoch 30/200 | Avg Loss: 7.8802
Epoch 40/200 | Avg Loss: 7.5162
Epoch 50/200 | Avg Loss: 9.5172
Epoch 60/200 | Avg Loss: 8.2833
Epoch 70/200 | Avg Loss: 5.6374
Epoch 80/200 | Avg Loss: 5.2613
Epoch 90/200 | Avg Loss: 6.2283
Epoch 100/200 | Avg Loss: 5.5353
Epoch 110/200 | Avg Loss: 4.9387
Epoch 120/200 | Avg Loss: 5.7900
Epoch 130/200 | Avg Loss: 4.5117

Early stopping triggered at Epoch 137!
Loss hasn't improved by more than 0.0001 for 25 consecutive epochs.
Training Complete!
Test Loss: 7.6741
[0.8496477881561856, 0.8371184371692011, 4032.206075949367, 606.2511021290547, 2.9017501953539817, 7.674064583912085, 2.7702102057266496, 2.0333102190041843, 9.125900160286909, 79, 7, 72, 58.12508033830539, 174.98986621050028, 191.57600117776943]


### HOUSE PRICES

In [7]:
X = house_price.drop('median_house_value', axis = 1)
y = house_price[['median_house_value']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train_scaled = y_train / 100000
y_test_scaled = y_test / 100000
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled.to_numpy(), dtype=torch.float32)
model_sigmoid = TwoHiddenLayerNNLeakyRelu(input_size=X.shape[1], hidden_size1=100, hidden_size2=50, output_size=y.shape[1])
model_sigmoid.trainLeakyRelu(X_train_tensor, y_train_tensor)
(OOS_predictions, OOS_loss) = model_sigmoid.testLeakyRelu(X_test_tensor, y_test_tensor)

y_test_numpy = y_test_scaled.to_numpy()
preds_OOS_numpy = OOS_predictions.cpu().numpy()
k = X_train_scaled.shape[1]
qof_OOS = get_qof(y_test_numpy, preds_OOS_numpy, k)
print(qof_OOS)

Epoch 10/200 | Avg Loss: 0.3341
Epoch 20/200 | Avg Loss: 0.3093
Epoch 30/200 | Avg Loss: 0.2917
Epoch 40/200 | Avg Loss: 0.2787
Epoch 50/200 | Avg Loss: 0.2709
Epoch 60/200 | Avg Loss: 0.2644
Epoch 70/200 | Avg Loss: 0.2587
Epoch 80/200 | Avg Loss: 0.2540
Epoch 90/200 | Avg Loss: 0.2496
Epoch 100/200 | Avg Loss: 0.2466
Epoch 110/200 | Avg Loss: 0.2423
Epoch 120/200 | Avg Loss: 0.2403
Epoch 130/200 | Avg Loss: 0.2393
Epoch 140/200 | Avg Loss: 0.2346
Epoch 150/200 | Avg Loss: 0.2325
Epoch 160/200 | Avg Loss: 0.2293
Epoch 170/200 | Avg Loss: 0.2270
Epoch 180/200 | Avg Loss: 0.2258
Epoch 190/200 | Avg Loss: 0.2228
Epoch 200/200 | Avg Loss: 0.2206
Training Complete!
Test Loss: 0.2810
[0.7944852783617966, 0.7939305146960247, 5589.046387258101, 1148.6313125003553, 0.5309168623466518, 0.28104509725969057, 0.530136866535134, 0.35730446581508934, 18.149491316902825, 4087, 12, 4075, 1312.7719365297658, -5163.38442788948, -5087.597630086709]


### INSURANCE

In [8]:
X = insurance.drop('charges', axis = 1)
y = insurance[['charges']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train_scaled = y_train / 6000
y_test_scaled = y_test / 6000
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled.to_numpy(), dtype=torch.float32)
model_sigmoid = TwoHiddenLayerNNLeakyRelu(input_size=X.shape[1], hidden_size1=100, hidden_size2=50, output_size=y.shape[1])
model_sigmoid.trainLeakyRelu(X_train_tensor, y_train_tensor)
(OOS_predictions, OOS_loss) = model_sigmoid.testLeakyRelu(X_test_tensor, y_test_tensor)

y_test_numpy = y_test_scaled.to_numpy()
preds_OOS_numpy = OOS_predictions.cpu().numpy()
k = X_train_scaled.shape[1]
qof_OOS = get_qof(y_test_numpy, preds_OOS_numpy, k)
print(qof_OOS)

Epoch 10/200 | Avg Loss: 0.6709
Epoch 20/200 | Avg Loss: 0.6090
Epoch 30/200 | Avg Loss: 0.6051
Epoch 40/200 | Avg Loss: 0.5770
Epoch 50/200 | Avg Loss: 0.5674
Epoch 60/200 | Avg Loss: 0.5711
Epoch 70/200 | Avg Loss: 0.5595
Epoch 80/200 | Avg Loss: 0.5249
Epoch 90/200 | Avg Loss: 0.5345
Epoch 100/200 | Avg Loss: 0.5280
Epoch 110/200 | Avg Loss: 0.4983
Epoch 120/200 | Avg Loss: 0.5106
Epoch 130/200 | Avg Loss: 0.4928
Epoch 140/200 | Avg Loss: 0.4899
Epoch 150/200 | Avg Loss: 0.4740
Epoch 160/200 | Avg Loss: 0.4715
Epoch 170/200 | Avg Loss: 0.4729
Epoch 180/200 | Avg Loss: 0.4542
Epoch 190/200 | Avg Loss: 0.4700
Epoch 200/200 | Avg Loss: 0.4476
Training Complete!
Test Loss: 0.6575
[0.847535282228363, 0.8428259473164978, 1155.7405566609807, 176.20965778855094, 0.8248309942453449, 0.657498723091608, 0.8108629496355151, 0.44534595272496813, 26.848203618458243, 268, 9, 259, 159.97263083073327, -94.37573847549046, -62.05685565089276]
